# SOXL官方产品页脚本：日级AUM/份额接口边界

仅访问已经归档页面指定的一份同发行人产品页脚本；精确字节NAS归档和只读字符串检查。不执行脚本、跟随接口、读取市场结果或联系供应商。

In [1]:
import datetime as dt,hashlib,io,json,re,shlex,subprocess,tarfile,tempfile
from pathlib import Path
from urllib.request import Request,build_opener,HTTPRedirectHandler
from urllib.error import HTTPError
from src.data.nas_market_data import nas_api_call,_settings,_ssh
root=Path.cwd();card=root/'research/stage1_soxl_official_product_chunk_source_probe_20261024';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
page=root/plan['source_page'];assert sha(page)==plan['source_page_sha256']
assert sha(root/plan['source_embedded_audit'])==plan['source_embedded_audit_sha256']
html=page.read_text(errors='replace')
assert html.count('src="'+plan['script_path_exact']+'"')==1
state_path=card/'run_state.json';state={'status':'started','plan_sha256':sha(plan_path),'GETs_started':1};state_path.write_text(json.dumps(state,indent=2)+'\n')
class NoRedirect(HTTPRedirectHandler):
 def redirect_request(self,req,fp,code,msg,headers,newurl):return None
opener=build_opener(NoRedirect);url=plan['script_url'];requested=dt.datetime.now(dt.timezone.utc)
status=None;headers={};body=b'';error=None
try:
 with opener.open(Request(url,headers={'User-Agent':'Mozilla/5.0 source qualification research','Accept':'text/javascript,application/javascript,*/*'}),timeout=30) as response:
  status=response.status;headers={k.lower():v for k,v in response.headers.items()};body=response.read(plan['request_budget']['max_response_bytes']+1)
except HTTPError as exc:
 status=exc.code;headers={k.lower():v for k,v in exc.headers.items()};body=exc.read(plan['request_budget']['max_response_bytes']+1);error='HTTPError '+str(exc.code)
except Exception as exc:error=type(exc).__name__+': '+str(exc)
received=dt.datetime.now(dt.timezone.utc)
raw=root/'data/snapshots/stage1_soxl_official_product_chunk_source_probe_20261024/product_chunk.js';raw.parent.mkdir(parents=True,exist_ok=True)
if body:raw.write_bytes(body)
if len(body)>plan['request_budget']['max_response_bytes']:error='response exceeds frozen byte budget'
receipt={'url':url,'requested_at_utc':requested.isoformat(),'received_at_utc':received.isoformat(),'http_status':status,'content_type':headers.get('content-type'),'response_date':headers.get('date'),'cache_control':headers.get('cache-control'),'last_modified':headers.get('last-modified'),'bytes':len(body),'sha256':hashlib.sha256(body).hexdigest() if body else None,'relative_path':str(raw.relative_to(root)) if body else None,'error':error}
text=body.decode('utf-8',errors='replace') if status==200 and body and not error else ''
contexts={}
for term in ['holdings','download','aum','shares','history','historical','flow']:
 matches=[];start=0
 for _ in range(3):
  i=text.lower().find(term,start)
  if i<0:break
  matches.append(text[max(0,i-90):min(len(text),i+130)])
  start=i+len(term)
 contexts[term]={'count':text.lower().count(term),'examples':matches}
url_like=list(dict.fromkeys(re.findall(r'https?://[^\s"\x27<>`]{5,240}',text)))[:50]
api_like=list(dict.fromkeys(re.findall(r'/api/[A-Za-z0-9_/?=&.%-]{3,180}',text)))[:50]
result={'status':'official_product_script_no_qualified_historical_AUM_source','plan_sha256':sha(plan_path),'source_page_sha256':sha(page),'official_script_receipt':receipt,'keyword_contexts':contexts,'url_like_strings':url_like,'api_like_paths':api_like,'historical_daily_SOXL_AUM_shares_qualified':False,'historical_first_public_vintages_qualified':False,'network_GETs':1,'market_outcome_reads':0,'model_fits':0,'accounts':0,'vendor_contacted':False,'limitation':'Only one archived-page-selected product chunk was fetched and read as text. Shared scripts or official dynamic endpoints may exist but were not requested; source-code strings are not historical source receipts.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
if body:
 with tempfile.TemporaryDirectory(prefix='soxl-product-js-archive-') as tmp:
  archive=Path(tmp)/'source.tar.gz'
  with tarfile.open(archive,'w:gz') as tar:
   info=tar.gettarinfo(str(raw),arcname='product_chunk.js');info.mtime=0
   with raw.open('rb') as stream:tar.addfile(info,stream)
   payload=json.dumps(receipt,ensure_ascii=False,sort_keys=True).encode();info=tarfile.TarInfo('receipt.json');info.size=len(payload);info.mtime=0;tar.addfile(info,io.BytesIO(payload))
  digest=hashlib.sha256(archive.read_bytes()).hexdigest();settings=_settings(root)
  nas=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
  assert nas['sha256']==digest and nas['size_bytes']==archive.stat().st_size
  remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
  assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
  restored=Path(tmp)/'restored.tar.gz'
  with restored.open('wb') as out:
   proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert proc.returncode==0 and hashlib.sha256(restored.read_bytes()).hexdigest()==digest
  (card/'nas_source_archive_receipt.json').write_text(json.dumps({'sha256':digest,'size_bytes':archive.stat().st_size,'nas_path':remote,'remote_hash_verified':True,'scoped_readback_verified':True},indent=2)+'\n')
state.update(status='completed',GETs_completed=1);state_path.write_text(json.dumps(state,indent=2)+'\n')
print({'status':status,'bytes':len(body),'error':error,'keyword_counts':{k:v['count'] for k,v in contexts.items()},'api_like_count':len(api_like)})


{'status': 200, 'bytes': 64769, 'error': None, 'keyword_counts': {'holdings': 26, 'download': 11, 'aum': 0, 'shares': 0, 'history': 1, 'historical': 0, 'flow': 3}, 'api_like_count': 0}


## 中文结论

# Direxion产品页脚本只暴露持仓弹窗，未给出历史AUM接口

从先前已归档的官方SOXL产品页中固定选取其唯一 `pages/product/[slug]` JavaScript资源，仅向Direxion同域请求一次。得到HTTP 200、**64,769字节** `application/javascript`，精确响应与UTC时间、SHA-256在`result.json`；原字节压缩包SHA-256 `cfb0fbd9e74d1f2e84064785c4f2d3122e1d4a2e7e51120a35bc2730403e3ae8` 已归档Yanxuan NAS、远端哈希核对并定向恢复。未执行脚本、跟随其引用或请求其他chunk/API；0市场结果、拟合、账户和供应商联系。

此产品页脚本的“Daily Fund Holdings”点击处理只是调用 **`show-daily-holdings` 模态窗**。文本含持仓/下载代码，但没有 AUM、份额字段、直接历史CSV URL或可直接识别的 `/api/` 历史路径；唯一出现的“history”是浏览器地址历史处理，不是基金时间序列。完整限定关键字上下文见`result.json`。

决定 `source_only_product_chunk_no_daily_historical_AUM_shares_endpoint_in_scope`。这只排除**本次所选产品页chunk**直接给出2011–2022日级AUM/份额与首发版本；共享chunk或发行人其他官方动态接口未读，不能说全站不存在。实际再平衡需求因子仍为 `needs_data`，不会把此前已失败的尾盘价格残差改名为基金流量重测。若继续追共享模块，必须另冻URL、次数、响应保存及首次公布门，再决定是否值得进入Stage1信息/账户检验。
